# st_pval — p-value как доля: False-Positive Psychology

**Статья:** Simmons, Nelson, Simonsohn (2011). *False-Positive Psychology: Undisclosed
Flexibility in Data Collection and Analysis Allows Presenting Anything as Significant.*
Psychological Science 22(11), 1359–1366. Один из самых цитируемых методологических текстов
десятилетия и спусковой крючок replication crisis.

**Узел дерева:** `st_pval` (🎲 Стата / гипотезы).

**Что строим:** свой permutation-test и t-test, Монте-Карло оценку false-positive rate,
четыре «степени свободы исследователя» из Table 1 и их комбинацию. Обучаемых моделей нет —
ядро статьи считается симуляцией, это и есть её протокол.

**Компьют:** CPU, все симуляции суммарно ~20 секунд.

**Parts:**
1. p-value руками — permutation test
2. t-test — аналитический ярлык
3. Монте-Карло машина FPR
4. Степени свободы исследователя A–D (Table 1)
5. Комбинированный p-hacker + ключевая фигура

## Эксперименты-первыми

Открой статью (она короткая и злая) и заполни от руки ДО того, как читать метод дальше:

- **Что реально показали** vs что обещает abstract:

- **Слабейший эксперимент / допущение:**

- **Где сломается первым** (границы метода):


## Зачем это

Порог $p<0.05$ объявляют защитой от ложных открытий: максимум 5% шансов увидеть эффект там,
где его нет. Simmons с коллегами показали, что гарантия держится только для анализа,
зафиксированного до сбора данных. Разреши себе обычные рабочие вольности — вторая зависимая
переменная, «доберу ещё участников», ковариата, выбор пары условий из трёх — и реальный шанс
ложного открытия вырастает до 61%.

Метод статьи: построить мир, где эффекта нет по построению, прогнать через него тысячи
«исследований» с этими вольностями и посчитать долю значимых. Преимущество перед словесной
критикой методологии — число, которое любой скептик воспроизводит у себя за минуту. Здесь
этим скептиком будешь ты.

Данные: нулевой мир генерируется — таков протокол статьи. Permutation-test дополнительно
проверяем на данных Стьюдента 1908 (прирост сна на двух снотворных) — реальном эксперименте,
с которого начался сам t-test.

## Теория (Фейнман)

### p-value руками

Есть две группы чисел, наблюдаем разницу средних $\Delta_{obs}$. Вопрос p-value: **если бы
разницы на самом деле не было, как часто случайность выдала бы $|\Delta| \ge |\Delta_{obs}|$?**

Перестановочный ответ буквален. Раз группы неразличимы, метки «группа 1 / группа 2» —
украшение: склей выборки, перемешай метки, пересчитай $\Delta$, повтори $N$ раз. Доля
перестановок с $|\Delta| \ge |\Delta_{obs}|$ и есть p-value. Определение целиком, без единой
формулы про распределения.

t-test — аналитический ярлык той же идеи: форма нулевого распределения известна заранее,

$$t = \frac{\bar x - \bar y}{s_p\sqrt{1/n_x + 1/n_y}},\qquad
s_p^2 = \frac{(n_x-1)s_x^2 + (n_y-1)s_y^2}{n_x+n_y-2},$$

и хвост считается по распределению Стьюдента с $n_x{+}n_y{-}2$ степенями свободы — перемешивать
ничего не нужно.

### Калибровка и её слом

При $H_0$ p-value распределено равномерно на $[0,1]$, поэтому правило «отвергаем при
$p<\alpha$» ошибается ровно в доле $\alpha$ нулевых миров. Это свойство процедуры, и оно
хрупкое: посчитай несколько p и отчитай минимальный — минимум равномерных величин уже не
равномерен:

$$P\big(\min(p_1,\dots,p_k)<\alpha\big) = 1-(1-\alpha)^k \quad \text{при независимости}.$$

При корреляции тестов инфляция меньше, но всё равно выше $\alpha$ — и формулой её уже не
посчитать. Вся статья — измерение этой инфляции симуляцией в четырёх реалистичных сценариях.

**Где у меня дыра:** честно ответь себе, что из этого ты бы вчера объяснила коллеге, не подсматривая.

❓ **Inline Question 0.** Исследователь проводит ровно ОДИН t-test на исследование, никакого
min-p, но публикует только значимые результаты, а остальные кладёт в стол. FPR отдельного
исследования по-прежнему 5%. Что при этом происходит с долей ложных результатов среди
ОПУБЛИКОВАННОГО? Чем этот механизм отличается от min-p и чем похож?

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Реальные данные: Student (1908), The probable error of a mean.
# Прирост сна (часы) на двух изомерах гиосциамина, n=10 пациентов на группу.
dextro = np.array([0.7, -1.6, -0.2, -1.2, -0.1, 3.4, 3.7, 0.8, 0.0, 2.0])
laevo  = np.array([1.9,  0.8,  1.1,  0.1, -0.1, 4.4, 5.5, 1.6, 4.6, 3.4])

ALPHA = 0.05
print("групповые средние:", dextro.mean(), laevo.mean())

## Part 1. p-value руками — permutation test

Реализуй двусторонний permutation-test для разницы средних. На данных Стьюдента должно
получиться $p \approx 0.08$: «почти значимо» — удобная точка, чтобы задавать неудобные вопросы.

Технические соглашения: сглаживание Лапласа $\frac{\#\{\text{экстремальнее или равно}\}+1}{N+1}$,
чтобы оценка не была нулём при конечном $N$.

In [ ]:
def perm_pvalue(x, y, n_perm=10000, rng=None):
    """Двусторонний permutation-test для разницы средних двух групп.

    Args:
        x, y: 1d float arrays — две независимые выборки.
        n_perm: int — число случайных перестановок.
        rng: np.random.Generator.
    Returns:
        float в (0, 1] — доля перестановок объединённой выборки, дающих
        |разницу средних| не меньше наблюдаемой; со сглаживанием +1/+1,
        поэтому ровно 0 не бывает.
    """
    rng = rng or np.random.default_rng(0)
    # YOUR CODE
    raise NotImplementedError

In [ ]:
pp = perm_pvalue(dextro, laevo, 20000, np.random.default_rng(0))
pt_ref = stats.ttest_ind(dextro, laevo).pvalue
assert 0 < pp <= 1, "p-value обязано лежать в (0,1] — проверь сглаживание"
assert abs(pp - pt_ref) < 0.02, (
    f"perm={pp:.4f} vs t-test={pt_ref:.4f}: на таких данных обе оценки одного хвоста "
    "должны сойтись; расходятся — чаще всего забыт модуль (двусторонность)")
p_shift = perm_pvalue(dextro + 10.0, laevo, 20000, np.random.default_rng(0))
assert p_shift < 0.001, "сдвиг +10 часов сна должен быть сверхзначим — проверь знак сравнения"
print(f"ok: perm p = {pp:.4f}  (Expected ≈ 0.080)")

❓ **Inline Question 1.** Сделай тест односторонним («лево-изомер лучше») — p упадёт вдвое,
до ~0.04, и результат «открытие». Кто и в какой момент должен фиксировать направление
гипотезы, чтобы это не было читерством? Как это связано с главной темой статьи?

## Part 2. t-test — аналитический ярлык

Тот же вопрос о хвосте, только распределение нуля известно в замкнутой форме. Реализуй
двухвыборочный t-test с объединённой дисперсией (формула — в теории выше) и сверь со scipy
до машинной точности.

In [ ]:
def t_pvalue(x, y):
    """Двухвыборочный t-test (pooled variance), двустороннее p.

    Args:
        x, y: 1d float arrays.
    Returns:
        float — p-value; совпадает со stats.ttest_ind(x, y).pvalue до ~1e-12.
    Note: хвост бери из stats.t (эта функция — единственное место, где нужна scipy).
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
pt = t_pvalue(dextro, laevo)
assert abs(pt - stats.ttest_ind(dextro, laevo).pvalue) < 1e-10, (
    "не сходится со scipy: проверь pooled-дисперсию (ddof=1) и степени свободы n_x+n_y-2")
assert abs(t_pvalue(dextro, laevo) - t_pvalue(laevo, dextro)) < 1e-12, "тест должен быть симметричен"
rng_t = np.random.default_rng(7)
big = t_pvalue(rng_t.standard_normal(500) + 1.0, rng_t.standard_normal(500))
assert big < 1e-10, "эффект в 1σ на n=500 должен быть сверхзначим"
print(f"ok: t p = {pt:.4f}  (Expected ≈ 0.0792)")

❓ **Inline Question 2.** Pooled-дисперсия предполагает равные дисперсии групп. Приведи
пример данных, где это допущение враньё, и скажи, чем его чинят (подсказка: Welch). Почему
при равных объёмах групп разница между вариантами почти незаметна?

## Part 3. Монте-Карло машина FPR

Ядро статьи — счётчик: сколько «нулевых» исследований объявили открытие. `experiment_fn(rng)`
возвращает p-value одного симулированного исследования (если внутри несколько анализов —
то p, которое отчитал бы автор). Дальше эта машина прогонит все сценарии Table 1.

Honest-бейзлайн: два условия по $n=20$ из $\mathcal N(0,1)$, один t-test. Ожидание — FPR
равен $\alpha$: тест калиброван по построению.

In [ ]:
def fpr_mc(experiment_fn, n_sims, rng, alpha=ALPHA):
    """Монте-Карло оценка false-positive rate процедуры.

    Args:
        experiment_fn: callable(rng) -> float — p-value одного нулевого исследования.
        n_sims: int — число симулированных исследований.
        rng: np.random.Generator — один на весь прогон.
        alpha: float — порог значимости.
    Returns:
        float — доля исследований с p < alpha.
    """
    # YOUR CODE
    raise NotImplementedError


def honest(rng, n=20):
    """Одно честное нулевое исследование: два условия N(0,1) по n, один t-test.

    Returns:
        float — p-value этого единственного теста.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_honest = fpr_mc(honest, 10000, np.random.default_rng(1))
assert 0.035 < r_honest < 0.065, (
    f"FPR честного теста = {r_honest:.3f}, ожидается ≈0.05: заметно больше — где-то течёт "
    "min-p или переиспользуется rng; меньше — p считается консервативно")
print(f"ok: honest FPR = {r_honest:.3f}  (Expected ≈ 0.050)")

❓ **Inline Question 3.** FPR≈α — свойство данных или процедуры? Что станет с FPR
честного теста при $n=200$ вместо $20$? А что при этом станет с мощностью (шансом поймать
реальный эффект)? Эти два ответа разные — почему?

## Part 4. Степени свободы исследователя (Table 1)

Четыре обычных вольности, каждая невинна на вид. Протоколы — из статьи:

- **A. Две зависимые переменные** (корреляция $r=0.5$): анализируешь DV1, DV2 и их среднее;
  в статью идёт лучший результат.
- **B. Добор наблюдений:** посмотрел на $n=20$ на группу; не значимо — добрал ещё 10 и
  посмотрел снова. Результат «значим», если хоть один взгляд дал $p<.05$.
- **C. Ковариата (пол):** t-test, ANCOVA с полом, модель с взаимодействием пол×условие;
  отчитывается лучший из эффектов условия/взаимодействия. — *дана готовой ниже: OLS-обвязка
  не входит в ядро, а паттерн min-p ты к тому моменту уже дважды напишешь сама.*
- **D. Три условия:** собрал три группы, отчитал любую значимую пару.

Симулируй A, B, D. Каждая функция — ОДНО нулевое исследование, возвращает то p, которое
отчитал бы автор.

In [ ]:
def dof_two_dvs(rng, n=20, r=0.5):
    """A: одно нулевое исследование с двумя коррелированными DV и гибким выбором результата.

    Args:
        rng: np.random.Generator; n: int — объём каждой из двух групп;
        r: float — корреляция DV внутри субъекта.
    Returns:
        float — p, которое отчитал бы автор (наихудшее для честности).
    Note: обе группы тянутся из одного и того же двумерного нулевого распределения.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_a = fpr_mc(dof_two_dvs, 10000, np.random.default_rng(2))
assert 0.085 < r_a < 0.125, (
    f"FPR(A) = {r_a:.3f}, ожидается ≈0.10 (в статье 9.5%): сильно ниже — потерялся один из "
    "трёх анализов; ~0.14 — DV сгенерированы независимыми, корреляция r=0.5 не попала в данные")
print(f"ok: A two-DVs FPR = {r_a:.3f}  (Expected ≈ 0.10, paper 0.095)")

In [ ]:
def dof_optional_stopping(rng, n1=20, add=10):
    """B: одно нулевое исследование с добором — взгляд на n1, при неудаче добор add и второй взгляд.

    Args:
        rng; n1: int — стартовый объём группы; add: int — добор на группу.
    Returns:
        float — p, которое отчитал бы автор.
    Note: добранные наблюдения — НОВЫЕ вытяжки, стартовые остаются в выборке.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_b = fpr_mc(dof_optional_stopping, 10000, np.random.default_rng(3))
assert 0.060 < r_b < 0.100, (
    f"FPR(B) = {r_b:.3f}, ожидается ≈0.08 (в статье 7.7%): ~0.05 — второй взгляд не влияет "
    "(перегенерила всю выборку заново?); ~0.10 — оба взгляда считаются независимыми")
print(f"ok: B optional stopping FPR = {r_b:.3f}  (Expected ≈ 0.080, paper 0.077)")

In [ ]:
def dof_three_conditions(rng, n=20):
    """D: одно нулевое исследование с тремя условиями и выбором пары.

    Args:
        rng; n: int — объём каждой из трёх групп.
    Returns:
        float — p, которое отчитал бы автор.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_d = fpr_mc(dof_three_conditions, 10000, np.random.default_rng(5))
assert 0.105 < r_d < 0.150, (
    f"FPR(D) = {r_d:.3f}, ожидается ≈0.126 (в статье 12.6%): проверь, что пар именно три "
    "и что группы не переиспользуют одни и те же сэмплы")
print(f"ok: D three conditions FPR = {r_d:.3f}  (Expected ≈ 0.126, paper 0.126)")

Сценарий C — готовый (обвязка). Прочитай `dof_covariate` и найди в нём тот же паттерн
min-p, который ты уже дважды написала сама.

In [ ]:
def ols_pvalues(X, y):
    # p-values коэффициентов OLS (двусторонние, t-распределение); дана готовой
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ X.T @ y
    resid = y - X @ beta
    df = len(y) - X.shape[1]
    sigma2 = resid @ resid / df
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    return 2 * stats.t.sf(np.abs(beta / se), df)


def dof_covariate(rng, n=20):
    # C: пол как ковариата — t-test, ANCOVA, взаимодействие; min p по условию/взаимодействию
    y = rng.standard_normal(2 * n)
    cond = np.r_[np.zeros(n), np.ones(n)]
    gender = np.tile(np.r_[np.zeros(n // 2), np.ones(n // 2)], 2)
    ps = [t_pvalue(y[cond == 0], y[cond == 1])]
    X1 = np.c_[np.ones(2 * n), cond, gender]
    ps.append(ols_pvalues(X1, y)[1])
    X2 = np.c_[np.ones(2 * n), cond, gender, cond * gender]
    p2 = ols_pvalues(X2, y)
    ps += [p2[1], p2[3]]
    return min(ps)


r_c = fpr_mc(dof_covariate, 10000, np.random.default_rng(4))
assert 0.090 < r_c < 0.135, f"FPR(C) = {r_c:.3f}, ожидается ≈0.113"
print(f"ok: C covariate FPR = {r_c:.3f}  (Expected ≈ 0.113, paper 0.117)")

❓ **Inline Question 4.** Для трёх независимых тестов $1-(1-.05)^3 = 14.3\%$, а у сценария A
вышло ~10%. Почему корреляция между анализами СНИЖАЕТ инфляцию? И почему «добор данных»
(сценарий B) ломает калибровку, хотя данных стало только больше?

## Part 5. Комбинированный p-hacker и ключевая фигура

Теперь всё разом, как в последней строке Table 1: три условия × три варианта DV (DV1, DV2,
среднее) × четыре анализа (t-test, ANCOVA, условие в модели с взаимодействием, само
взаимодействие) × двухстадийный добор. 36 анализов на стадию; вторая стадия — только если
первая ничего не дала. Заявка статьи: **60.7%**.

In [ ]:
def hacker_combined(rng, n1=20, add=10, r=0.5):
    """Все четыре свободы одновременно — одно нулевое «исследование» недобросовестного автора.

    Args:
        rng; n1: int — стартовый объём каждой из трёх групп; add: int — добор;
        r: float — корреляция двух DV.
    Returns:
        float — p, которое отчитал бы автор.
    Note: пол внутри группы сбалансирован; добор — новые вытяжки к старым.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
r_comb = fpr_mc(hacker_combined, 4000, np.random.default_rng(6))
assert 0.55 < r_comb < 0.66, (
    f"FPR(combined) = {r_comb:.3f}, заявка статьи 60.7%: сильно ниже — какая-то из свобод "
    "не участвует (чаще всего забывают варианты DV или вторую стадию); выше — min берётся "
    "по лишним коэффициентам OLS (интерсепт, пол)")
assert r_comb > 8 * r_honest, "комбинированный эффект должен качественно отличаться от честных 5%"
print(f"ok: combined FPR = {r_comb:.3f}  (Expected ≈ 0.60, paper 0.607)")

### Ключевая фигура — своя Table 1

In [ ]:
labels = ["честный", "A: 2 DV", "B: добор", "C: ковариата", "D: 3 условия", "A+B+C+D"]
mine  = [r_honest, r_a, r_b, r_c, r_d, r_comb]
paper = [0.05, 0.095, 0.077, 0.117, 0.126, 0.607]

fig, ax = plt.subplots(figsize=(8, 4))
xs = np.arange(len(labels))
ax.bar(xs, mine, width=0.6, label="моя симуляция")
ax.scatter(xs, paper, zorder=3, marker="D", label="Simmons et al., Table 1")
ax.axhline(ALPHA, ls="--", lw=1)
ax.set_xticks(xs, labels, rotation=20)
ax.set_ylabel("false-positive rate")
ax.legend(); ax.set_title("Инфляция FPR степенями свободы исследователя")
plt.tight_layout(); plt.show()

for l, m, p in zip(labels, mine, paper):
    print(f"{l:14s} моя={m:5.3f}  статья={p:5.3f}  Δ={m-p:+.3f}")

Каждый столбик обязан накрыть ромбик с точностью до Монте-Карло шума. Пунктир на 0.05 —
обещание, которое даёт «$p<.05$». Комбинированный столбик показывает, чего это обещание
стоит без фиксации анализа: подбрасывание монетки было бы честнее — у монетки хотя бы 50%.

❓ **Inline Question 5 (узнавание).** Ты сравнила 12 вариантов модели на одном тест-сете,
лучший «значимо» обошёл бейзлайн. Сколько p-values ты молча минимизировала? В какой ещё
ситуации из твоей практики стоит прогнать вот такую симуляцию нулевого мира, прежде чем
верить своему открытию?

### Заметка про хрупкий узел

Самая частая ошибка в B и в комбинированном сценарии — при доборе перегенерировать выборку
целиком. Это другой протокол: два независимых исследования вместо подглядывания, и FPR у него
другой. Добор = старые наблюдения остаются, новые дотягиваются тем же rng.

## Рефлексия

- **Что забираю в работу** (один конкретный приём):

- **В какой НОВОЙ ситуации потянусь за симуляцией нулевого мира:**

- **Что осталось мутным** → кандидат в следующий узел: `st_power` (сколько данных, чтобы
  поймать реальный эффект), `st_mult` (Bonferroni/FDR — законная починка min-p), `st_ci`
  (bootstrap — интервал вместо вердикта):


## Bonus: Going deeper (без баллов)

1. Прикрути к `hacker_combined` поправку Бонферрони на все 72 анализа. Вернулся ли FPR к 5%?
   Прикинь, что стало бы с мощностью при реальном эффекте — и почему Бонферрони тут почти
   бесплатен, а в геномике (миллион тестов) — катастрофа.
2. Публикационный фильтр: авторы честны (один тест), но журнал печатает только $p<.05$.
   При доле истинных эффектов 10% и мощности 0.5 посчитай долю ложных среди опубликованных.
3. Законный добор: alpha-spending (Pocock). Реализуй два взгляда с порогом 0.0294 на каждом
   и покажи симуляцией, что суммарный FPR ≈ 5%.

**Лог экспериментов:**


---
## SPOILERS

Полные решения. Начни сама; сюда — когда застряла всерьёз или для сверки после.

<details><summary>Part 1: perm_pvalue</summary>

```python
def perm_pvalue(x, y, n_perm=10000, rng=None):
    rng = rng or np.random.default_rng(0)
    obs = abs(x.mean() - y.mean())
    z = np.concatenate([x, y])
    cnt = 0
    for _ in range(n_perm):
        p = rng.permutation(z)
        cnt += abs(p[:len(x)].mean() - p[len(x):].mean()) >= obs
    return (cnt + 1) / (n_perm + 1)
```
</details>

<details><summary>Part 2: t_pvalue</summary>

```python
def t_pvalue(x, y):
    nx, ny = len(x), len(y)
    sp2 = ((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2)
    t = (x.mean() - y.mean()) / np.sqrt(sp2 * (1 / nx + 1 / ny))
    return 2 * stats.t.sf(abs(t), nx + ny - 2)
```
</details>

<details><summary>Part 3: fpr_mc + honest</summary>

```python
def fpr_mc(experiment_fn, n_sims, rng, alpha=ALPHA):
    hits = 0
    for _ in range(n_sims):
        hits += experiment_fn(rng) < alpha
    return hits / n_sims

def honest(rng, n=20):
    return t_pvalue(rng.standard_normal(n), rng.standard_normal(n))
```
</details>

<details><summary>Part 4: A, B, D</summary>

```python
def dof_two_dvs(rng, n=20, r=0.5):
    cov = [[1, r], [r, 1]]
    a = rng.multivariate_normal([0, 0], cov, n)
    b = rng.multivariate_normal([0, 0], cov, n)
    return min(t_pvalue(a[:, 0], b[:, 0]), t_pvalue(a[:, 1], b[:, 1]),
               t_pvalue(a.mean(1), b.mean(1)))

def dof_optional_stopping(rng, n1=20, add=10):
    x, y = rng.standard_normal(n1), rng.standard_normal(n1)
    p1 = t_pvalue(x, y)
    if p1 < 0.05:
        return p1
    x2 = np.concatenate([x, rng.standard_normal(add)])
    y2 = np.concatenate([y, rng.standard_normal(add)])
    return min(p1, t_pvalue(x2, y2))

def dof_three_conditions(rng, n=20):
    g = [rng.standard_normal(n) for _ in range(3)]
    return min(t_pvalue(g[i], g[j]) for i in range(3) for j in range(i + 1, 3))
```
</details>

<details><summary>Part 5: hacker_combined</summary>

```python
def hacker_combined(rng, n1=20, add=10, r=0.5):
    cov = [[1, r], [r, 1]]

    def stage_minp(groups, genders):
        ps = []
        for i in range(3):
            for j in range(i + 1, 3):
                gi, gj = groups[i], groups[j]
                for dv_i, dv_j in [(gi[:, 0], gj[:, 0]), (gi[:, 1], gj[:, 1]),
                                   (gi.mean(1), gj.mean(1))]:
                    y = np.concatenate([dv_i, dv_j])
                    cnd = np.r_[np.zeros(len(dv_i)), np.ones(len(dv_j))]
                    gen = np.concatenate([genders[i], genders[j]])
                    ps.append(t_pvalue(dv_i, dv_j))
                    X1 = np.c_[np.ones(len(y)), cnd, gen]
                    ps.append(ols_pvalues(X1, y)[1])
                    X2 = np.c_[np.ones(len(y)), cnd, gen, cnd * gen]
                    p2 = ols_pvalues(X2, y)
                    ps += [p2[1], p2[3]]
        return min(ps)

    groups = [rng.multivariate_normal([0, 0], cov, n1) for _ in range(3)]
    genders = [np.r_[np.zeros(n1 // 2), np.ones(n1 // 2)] for _ in range(3)]
    p1 = stage_minp(groups, genders)
    if p1 < 0.05:
        return p1
    groups = [np.vstack([g, rng.multivariate_normal([0, 0], cov, add)]) for g in groups]
    genders = [np.r_[g, np.zeros(add // 2), np.ones(add - add // 2)] for g in genders]
    return min(p1, stage_minp(groups, genders))
```
</details>